# BigQuery + DVC Data Extraction and EDA

This notebook demonstrates how to pull the three pipeline tables from BigQuery, version them with DVC, and perform exploratory data analysis on `category_tree`, `events`, and `item_properties`.

## Setup

The notebook uses the `BigQueryQuery` helper from `ecommerce-recommender/data-pipeline/bigquery_query.py` to run queries, write CSV exports, and add them to DVC. Make sure your `.venv` is active and DVC is installed.

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd

from data_pipeline.bigquery_query import BigQueryQuery

## Configure the Extractor

Set the BigQuery project and dataset values, then create an output folder for the exported CSVs.

In [5]:
PROJECT_ID = "retail-rocket-498618"  # update to your project id
DATASET_ID = "retailrocket"  # update to your dataset id
OUTPUT_DIR = Path('data/exports')
DVC_REPO_PATH = Path('..')

query_client = BigQueryQuery(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    output_dir=OUTPUT_DIR,
    dvc_repo_path=DVC_REPO_PATH,
)

/home/dpaula/code/MLENG_FIAP/fase_2/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## Extract the Three Pipeline Tables

Export the tables to local CSV files and version them with DVC. If the tables are large, the notebook will stream the results row by row.

In [8]:
category_tree_csv = query_client.extract_table(
    table_name='retail-rocket_category_tree',
    destination_name='category_tree.csv',
)
events_csv = query_client.extract_table(
    table_name='retail-rocket_events',
    destination_name='events.csv',
)
item_properties_csv = query_client.extract_table(
    table_name='retail-rocket_item_properties',
    destination_name='item_properties.csv',
)

category_tree_csv, events_csv, item_properties_csv

RuntimeError: DVC add failed for data/exports/category_tree.csv: ERROR: you are not inside of a DVC repository (checked up to mount point '/')

## Load Extracted CSVs

Read the exported files into pandas DataFrames for analysis.

In [ ]:
df_category_tree = pd.read_csv(category_tree_csv)
df_events = pd.read_csv(events_csv)
df_item_properties = pd.read_csv(item_properties_csv)

df_category_tree.head(), df_events.head(), df_item_properties.head()

## Category Tree EDA

Inspect the hierarchical structure and count categories.

In [ ]:
print('Category tree rows:', len(df_category_tree))
print('Unique category IDs:', df_category_tree['category_id'].nunique())
print('Top parent categories:')
print(df_category_tree['parent_id'].value_counts().head(10))

## Events EDA

Analyze event counts, unique users, and event types.

In [ ]:
print('Total events:', len(df_events))
print('Unique users:', df_events['user_id'].nunique())

if 'event' in df_events.columns:
    print('Event type counts:')
    print(df_events['event'].value_counts().head(10))
else:
    print('No event type column in events table.')

## Item Properties EDA

Understand how many unique items and properties exist in the merged properties table.

In [ ]:
print('Total item properties rows:', len(df_item_properties))
print('Unique items:', df_item_properties['item_id'].nunique())
print('Sample properties:')
print(df_item_properties.head(10))

## Notes

- The exported CSV files are versioned with DVC through `dvc add`.
- If the dataset is large, use filtered queries or `LIMIT` to reduce output size for exploratory analysis.
- Update `PROJECT_ID` and `DATASET_ID` to match your environment before running.